# Une question à ChatGPT, ça consomme combien ?

Projet 4 – Comparateur de modèles d'IA frugale.

On compare 3 IA sur la même tâche, repérer des SMS arnaques, et on mesure les performances, le temps et l'énergie avec CodeCarbon.

Pour les mesures finales : tout lancer en une fois sur le même PC, branché sur secteur, avec les autres applis fermées.

**Version 2 (9 nov.)** : les cases modifiées portent la mention ✏️ v2.
- Régression logistique : test `class_weight="balanced"` choisi par validation croisée.
- Mesures plus longues (au moins une dizaine de secondes) et échauffement avant les mesures de prédiction.
- Énergie du processeur seule (vraie mesure) en plus de l'énergie totale.
- Mini-chatbot et DistilBERT chargés en float32, avec estimation de la durée avant de tout lancer.
- Si le chatbot est testé sur une partie des SMS, les 3 IA sont comparées sur les mêmes SMS.
- Graphe : ligne de référence « IA qui répond toujours normal » et source exacte du chiffre ChatGPT.

## 1. Installation

In [ ]:
# ✏️ v2 : installation à faire UNE seule fois (déjà fait sur le PC d'Anis).
# Sur un autre PC, enlevez le « # » au début de la ligne suivante.
# !pip install codecarbon transformers torch scikit-learn pandas matplotlib

## 2. Imports et paramètres

In [ ]:
import time
import pandas as pd
import matplotlib.pyplot as plt
import torch
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate  # ✏️ v2
from sklearn.pipeline import make_pipeline  # ✏️ v2
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForCausalLM
from codecarbon import OfflineEmissionsTracker

In [ ]:
URL_DONNEES = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"  # copie du fichier officiel UCI
NOM_BERT = "distilbert-base-uncased"
NOM_LLM = "Qwen/Qwen2.5-0.5B-Instruct"
NB_TEST_LLM = None
ESTIMATION_CHATGPT_WH = 340  # 0,34 Wh par question (Sam Altman, juin 2025) × 1 000
PAYS = "FRA"
torch.manual_seed(42)

## 3. Les données

SMS Spam Collection (Almeida, Gómez Hidalgo et Yamakami, 2011) : 5 574 SMS en anglais, étiquetés ham (normal) ou spam.

On enlève d'abord les SMS en double : sinon une centaine de SMS du test se retrouveraient aussi dans l'entraînement, et les scores seraient trop beaux.

Ensuite : 80 % pour entraîner, 20 % pour tester, `random_state = 42`. Les 3 IA sont testées sur les mêmes SMS.

In [ ]:
def charger_donnees(url):
    data = pd.read_csv(url, sep="\t", header=None, names=["label", "sms"], quoting=3)
    data["label"] = (data["label"] == "spam").astype(int)
    return data.drop_duplicates(subset="sms").reset_index(drop=True)

In [ ]:
data = charger_donnees(URL_DONNEES)
print(len(data), "SMS uniques dont", data["label"].sum(), "spams")

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    data["sms"].tolist(), data["label"].tolist(), test_size=0.2, stratify=data["label"], random_state=42
)
print(len(x_train), "SMS pour entraîner,", len(x_test), "SMS pour tester")

## 4. Mesurer le temps et l'énergie

CodeCarbon mesure l'énergie consommée pendant le calcul et la convertit en CO₂ avec l'électricité française.
Sur le PC d'Anis (Windows 11), le **processeur est vraiment mesuré** (Windows EMI). La **mémoire est estimée** à 10 W fixes : l'énergie totale = processeur (mesuré) + mémoire (estimée). ✏️ v2 : on garde aussi l'énergie du processeur seule.

Un calcul très court donne une mesure trop petite : on le répète plusieurs fois et on divise. ✏️ v2 : chaque mesure doit durer **au moins une dizaine de secondes**.

✏️ v2 : avant de mesurer une prédiction, on la lance une fois sans mesurer (« échauffement »), pour ne pas compter les coûts de démarrage.

On mesure séparément l'entraînement et l'inférence (prédire les SMS de test). L'énergie et le CO₂ de l'inférence sont ramenés à 1 000 SMS.

In [ ]:
def mesurer(fonction, repetitions=1):
    tracker = OfflineEmissionsTracker(country_iso_code=PAYS, measure_power_secs=1, log_level="error", save_to_file=False)
    tracker.start()
    debut = time.perf_counter()
    for i in range(repetitions):
        resultat = fonction()
    duree = time.perf_counter() - debut
    co2 = tracker.stop()
    donnees = tracker.final_emissions_data
    energie = donnees.energy_consumed      # total en kWh : processeur (mesuré) + mémoire (estimée)
    energie_cpu = donnees.cpu_energy       # ✏️ v2 : processeur seul, en kWh (vraie mesure)
    return resultat, duree / repetitions, energie / repetitions, co2 / repetitions, energie_cpu / repetitions

In [ ]:
def ligne(nom, y_vrai, mesure, nb_sms):
    predictions, duree, energie, co2, energie_cpu = mesure  # ✏️ v2 : + énergie du processeur
    facteur = 1000 / nb_sms
    return {
        "modele": nom,
        "accuracy": accuracy_score(y_vrai, predictions),
        "precision": precision_score(y_vrai, predictions, zero_division=0),
        "recall": recall_score(y_vrai, predictions, zero_division=0),
        "f1": f1_score(y_vrai, predictions, zero_division=0),
        "temps_par_sms_ms": duree / nb_sms * 1000,
        "energie_1000_sms_Wh": energie * 1000 * facteur,
        "energie_cpu_1000_sms_Wh": energie_cpu * 1000 * facteur,  # ✏️ v2
        "co2_1000_sms_g": co2 * 1000 * facteur,
    }

## 5. IA n°1 : régression logistique (TF-IDF)

Le modèle le plus simple : il regarde quels mots apparaissent dans le SMS.

In [ ]:
def entrainer_logreg(x, y, class_weight=None):  # ✏️ v2 : option class_weight
    vect = TfidfVectorizer()
    modele = LogisticRegression(max_iter=1000, class_weight=class_weight)
    modele.fit(vect.fit_transform(x), y)
    return vect, modele


def predire_logreg(vect, modele, x):
    return modele.predict(vect.transform(x)).tolist()

✏️ v2 : **normale ou `class_weight="balanced"` ?**

Il n'y a qu'environ 13 % de spams. Avec `class_weight="balanced"`, l'IA accorde plus d'importance aux spams pour en trouver davantage.
On choisit la meilleure version **sans regarder les SMS de test** (sinon ce serait tricher) : validation croisée sur les SMS d'entraînement, coupés en 5 parts. Le critère est le F1.

In [ ]:
validation = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
lignes_cv = []
for version in [None, "balanced"]:
    chaine = make_pipeline(TfidfVectorizer(), LogisticRegression(max_iter=1000, class_weight=version))
    res = cross_validate(chaine, x_train, y_train, cv=validation, scoring=["recall", "precision", "f1"])
    lignes_cv.append({"version": version or "normale",
                      "recall": res["test_recall"].mean(),
                      "precision": res["test_precision"].mean(),
                      "f1": res["test_f1"].mean()})

choix_cv = pd.DataFrame(lignes_cv).set_index("version")
CLASS_WEIGHT = "balanced" if choix_cv.loc["balanced", "f1"] > choix_cv.loc["normale", "f1"] else None
print("Version gardée pour la mesure :", CLASS_WEIGHT or "normale")
choix_cv.round(3)

In [ ]:
# ✏️ v2 : 200 et 2 000 répétitions pour que chaque mesure dure au moins une dizaine de secondes
mesure_ent_logreg = mesurer(lambda: entrainer_logreg(x_train, y_train, CLASS_WEIGHT), repetitions=200)
vect, logreg = mesure_ent_logreg[0]

predire_logreg(vect, logreg, x_test)  # ✏️ v2 : échauffement (non mesuré)
mesure_logreg = mesurer(lambda: predire_logreg(vect, logreg, x_test), repetitions=2000)
resultats = [ligne("Régression logistique", y_test, mesure_logreg, len(x_test))]

print("Durée des mesures : entraînement", round(mesure_ent_logreg[1] * 200, 1), "s, prédiction", round(mesure_logreg[1] * 2000, 1), "s")
print(resultats[-1])

## 6. IA n°2 : DistilBERT (66 millions de paramètres)

On part de DistilBERT, pré-entraîné sur du texte général, et on l'adapte aux SMS avec notre jeu d'entraînement.

On n'utilise pas un modèle déjà entraîné sur ces SMS : il aurait pu voir nos SMS de test.

Sur un PC sans GPU, l'entraînement peut être long : mettez `epochs=1` si besoin.

In [ ]:
def lots(liste, taille):
    return [liste[i:i + taille] for i in range(0, len(liste), taille)]


def charger_bert(nom):
    tok = AutoTokenizer.from_pretrained(nom)
    modele = AutoModelForSequenceClassification.from_pretrained(nom, num_labels=2, dtype=torch.float32)  # ✏️ v2
    return tok, modele


def entrainer_bert(tok, modele, x, y, epochs=2):
    optim = torch.optim.AdamW(modele.parameters(), lr=5e-5)
    modele.train()
    for e in range(epochs):
        total = 0
        for xb, yb in zip(lots(x, 32), lots(y, 32)):
            entree = tok(xb, padding=True, truncation=True, max_length=64, return_tensors="pt")
            perte = modele(**entree, labels=torch.tensor(yb)).loss
            perte.backward()
            optim.step()
            optim.zero_grad()
            total += perte.item()
        print("epoch", e + 1, "perte moyenne", round(total / len(lots(x, 32)), 4))
    modele.eval()


def predire_bert(tok, modele, x):
    predictions = []
    with torch.no_grad():
        for xb in lots(x, 32):
            entree = tok(xb, padding=True, truncation=True, max_length=64, return_tensors="pt")
            predictions += modele(**entree).logits.argmax(dim=1).tolist()
    return predictions

In [ ]:
tok_bert, bert = charger_bert(NOM_BERT)
mesure_ent_bert = mesurer(lambda: entrainer_bert(tok_bert, bert, x_train, y_train))

predire_bert(tok_bert, bert, x_test[:64])  # ✏️ v2 : échauffement (non mesuré)
mesure_bert = mesurer(lambda: predire_bert(tok_bert, bert, x_test), repetitions=3)
resultats.append(ligne("DistilBERT", y_test, mesure_bert, len(x_test)))
print(resultats[-1])

## 7. IA n°3 : mini-chatbot (Qwen2.5, 0,5 milliard de paramètres)

Un petit LLM généraliste, comme un ChatGPT en miniature. On lui pose la question pour chaque SMS, sans l'entraîner.

Le téléchargement fait environ 1 Go. Si c'est trop long, mettez `NB_TEST_LLM = 200` : l'énergie reste ramenée à 1 000 SMS.

In [ ]:
def charger_llm(nom):
    tok = AutoTokenizer.from_pretrained(nom)
    modele = AutoModelForCausalLM.from_pretrained(nom, dtype=torch.float32)  # ✏️ v2 : float32, le format normal sur processeur
    modele.eval()
    return tok, modele


def demander(tok, modele, sms):
    messages = [{"role": "user", "content": "Is this SMS a spam or a scam? Answer only yes or no.\nSMS: " + sms}]
    texte = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    entree = tok(texte, return_tensors="pt")
    with torch.no_grad():
        sortie = modele.generate(**entree, max_new_tokens=3, do_sample=False)
    reponse = tok.decode(sortie[0][entree["input_ids"].shape[1]:], skip_special_tokens=True)
    return 1 if "yes" in reponse.lower() else 0


def predire_llm(tok, modele, x):
    return [demander(tok, modele, sms) for sms in x]

In [ ]:
tok_llm, llm = charger_llm(NOM_LLM)

Test rapide sur 2 SMS avant de tout lancer (on attend 0 puis 1).
✏️ v2 : ce test sert aussi d'échauffement et donne une estimation de la durée totale. Si c'est trop long, mettez `NB_TEST_LLM = 200` dans les paramètres.

In [ ]:
debut = time.perf_counter()
print(demander(tok_llm, llm, "Hey, are we still meeting at the library at 6?"))
print(demander(tok_llm, llm, "URGENT! Your parcel is blocked. Pay 1.99 GBP now at this link to receive it"))
par_sms = (time.perf_counter() - debut) / 2
print(f"≈ {par_sms:.1f} s par SMS → ≈ {par_sms * len(x_test) / 60:.0f} min pour les {len(x_test)} SMS de test")  # ✏️ v2

In [ ]:
x_llm = x_test[:NB_TEST_LLM]
y_llm = y_test[:NB_TEST_LLM]
mesure_llm = mesurer(lambda: predire_llm(tok_llm, llm, x_llm))
resultats.append(ligne("Mini-chatbot (Qwen 0,5B)", y_llm, mesure_llm, len(x_llm)))
print(resultats[-1])

In [ ]:
# ✏️ v2 : si NB_TEST_LLM n'est pas None, le chatbot n'est noté que sur une partie des SMS.
# Pour une comparaison juste, on note alors les 3 IA sur exactement les mêmes SMS.
if NB_TEST_LLM is not None:
    n = len(x_llm)
    memes_sms = pd.DataFrame([
        {"modele": "Régression logistique", "accuracy": accuracy_score(y_llm, mesure_logreg[0][:n]), "f1": f1_score(y_llm, mesure_logreg[0][:n], zero_division=0)},
        {"modele": "DistilBERT", "accuracy": accuracy_score(y_llm, mesure_bert[0][:n]), "f1": f1_score(y_llm, mesure_bert[0][:n], zero_division=0)},
        {"modele": "Mini-chatbot (Qwen 0,5B)", "accuracy": accuracy_score(y_llm, mesure_llm[0]), "f1": f1_score(y_llm, mesure_llm[0], zero_division=0)},
    ])
    print("Scores des 3 IA sur les", n, "mêmes SMS :")
    display(memes_sms)
else:
    print("Le chatbot a été noté sur tous les SMS de test : rien à faire.")

## 8. Tableau des résultats (inférence)

- `accuracy` : % de SMS bien classés
- `precision` : parmi les SMS classés spam, % qui sont vraiment des spams
- `recall` : parmi les vrais spams, % que l'IA a trouvés
- `f1` : moyenne harmonique de precision et recall (✏️ v2 : « harmonique », pas une moyenne simple)
- `temps_par_sms_ms` : temps pour prédire un SMS
- `energie_1000_sms_Wh` et `co2_1000_sms_g` : pour 1 000 SMS (processeur mesuré + mémoire estimée)
- ✏️ v2 : `energie_cpu_1000_sms_Wh` : processeur seul, la partie vraiment mesurée
- `x_energie_vs_logreg` : combien de fois plus d'énergie que la régression logistique
- `recharges_smartphone` : l'énergie en recharges de téléphone (0,022 kWh par recharge, valeur de l'EPA utilisée par Luccioni et al., 2024)

⚠️ Environ 87 % des SMS sont normaux : une IA qui répondrait toujours « normal » aurait déjà 87 % d'accuracy. Regardez aussi le `f1`.

In [ ]:
def tableau(resultats):
    df = pd.DataFrame(resultats)
    df["x_energie_vs_logreg"] = df["energie_1000_sms_Wh"] / df["energie_1000_sms_Wh"][0]
    df["recharges_smartphone"] = df["energie_1000_sms_Wh"] / 22
    return df

In [ ]:
pd.set_option("display.float_format", "{:.3g}".format)
df = tableau(resultats)
df

## 9. Graphe exactitude vs énergie

C'est le graphe central du poster. L'axe de l'énergie est en échelle log parce que les écarts peuvent être énormes.

✏️ v2 : la ligne rouge pointillée, c'est l'exactitude d'une IA qui répondrait toujours « normal » (environ 87 %). Une IA utile doit être nettement au-dessus.

La ligne grise, c'est ChatGPT : 0,34 Wh par question, chiffre donné par Sam Altman (PDG d'OpenAI) dans son billet « The Gentle Singularity » (juin 2025), sans méthode publiée, × 1 000. Ce n'est pas une mesure, et une question à ChatGPT n'est pas la même tâche que classer un SMS : c'est un ordre de grandeur.

In [ ]:
def tracer(df, estimation_chatgpt, reference):  # ✏️ v2 : + ligne de référence
    plt.figure(figsize=(8, 5))
    for i in range(len(df)):
        plt.scatter(df["energie_1000_sms_Wh"][i], df["accuracy"][i] * 100, s=120)
        plt.annotate(df["modele"][i], (df["energie_1000_sms_Wh"][i], df["accuracy"][i] * 100), textcoords="offset points", xytext=(8, 8))
    plt.axvline(estimation_chatgpt, color="gray", linestyle="--", label="ChatGPT : 0,34 Wh/question (selon S. Altman, non mesuré)")
    plt.axhline(reference, color="red", linestyle=":", label="IA qui répond toujours « normal »")  # ✏️ v2
    plt.xscale("log")
    plt.margins(x=0.2, y=0.15)
    plt.ylim(min(plt.ylim()[0], reference - 3), plt.ylim()[1])
    plt.legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), fontsize=9)  # ✏️ v2 : légende sous le graphe, sans chevauchement
    plt.xlabel("Énergie pour 1 000 SMS (Wh, échelle log)")
    plt.ylabel("Exactitude (accuracy, % de SMS bien classés)")
    plt.title("Exactitude vs énergie des 3 IA")
    plt.grid(True, alpha=0.3)
    plt.savefig("exactitude_vs_energie.png", dpi=200, bbox_inches="tight")
    plt.show()

In [ ]:
reference = (1 - sum(y_test) / len(y_test)) * 100  # ✏️ v2 : % de SMS normaux dans le test
tracer(df, ESTIMATION_CHATGPT_WH, reference)

## 10. Le coût de l'entraînement

Entraînement des IA n°1 et n°2. Ne sont pas comptés, et c'est une limite à mettre sur le poster :
- l'entraînement du chatbot, fait par son créateur ;
- ✏️ v2 : le pré-entraînement de DistilBERT, fait par Hugging Face. On ne mesure que notre adaptation aux SMS.

In [ ]:
entrainement = pd.DataFrame([
    {"modele": "Régression logistique", "energie_entrainement_Wh": mesure_ent_logreg[2] * 1000, "temps_entrainement_s": mesure_ent_logreg[1]},
    {"modele": "DistilBERT", "energie_entrainement_Wh": mesure_ent_bert[2] * 1000, "temps_entrainement_s": mesure_ent_bert[1]},
])
entrainement

## 11. Sauvegarde

Le tableau et le graphe sont enregistrés pour le poster et le GitHub.

In [ ]:
df.to_csv("comparison.csv", index=False)
entrainement.to_csv("entrainement.csv", index=False)
print("Fichiers enregistrés : comparison.csv, entrainement.csv, exactitude_vs_energie.png")